
# Meander - LSM-1 advection-diffusion (k-eps model)

In this example we compute the advection-diffusion of particles in a meandering 
channel with the LSM-1 model with k-eps diffusion.


In [ ]:
import cylag
import os
import sys
import numpy as np
import matplotlib.pylab as plt

#sphinx_gallery_thumbnail_number = 1

## Define EulerianFieldSet




In [ ]:
file_name = os.path.join('data', 'r2d_meander.slf')
bnd_file = os.path.join('data', 'geo_meander.cli')

fset = cylag.EulerianFieldSet.from_telemac2d(
    file_name,
    bnd_file=bnd_file, 
    optional_fields=['TURBULENT ENERG.', 'DISSIPATION'])

## Define LagrangianParticleSet




In [ ]:
poly = np.array([[-6.085,-0.270],\
                 [-5.800,-0.750],\
                 [-4.958,-0.260],\
                 [-5.235,0.220]])

position = cylag.init_positions_2d(poly=poly, grid_res=(50, 50))
npart = position.shape[0]

print('Number of particles = ', npart)

pset = cylag.LagrangianParticleSet(\
    position, 
    dim=2)

## Set general parameters




In [ ]:
parameters = {
    'final_time': 160.,
    'time_step': 0.1,
    'frozen_eulerian_fields': True,
    'model': 1,
    'time_scheme': 1,
    'diffusion_model': 2,
    'schmidt_number': 0.72,
    'listing': True,
    'listing_printout_period': 100,
    'output_file': True,
    'output_printout_period': 100,
    'output_file_format': 'txt',
    'output_rep': 'resu_03',
    'output_file_name': 'particles_2d',
    'boundary_conditions_type': 1,
    'rebound_damping_coef': 0.,
    }

## Run cylag




In [ ]:
cylag_solver = cylag.Solver(fset, pset, parameters)
cylag_solver.solve()

## Post-processing

Load particles :



In [ ]:
part = cylag.ParticlesIO.from_cylag_txt('resu_03/particles_2d.txt')

rec = len(part.times) - 1
time = part.times[rec]

Field set from telemac results :



In [ ]:
file_name = os.path.join('data', 'r2d_meander.slf')
bnd_file = os.path.join('data', 'geo_meander.cli')
fset = cylag.EulerianFieldSet.from_telemac2d(file_name, bnd_file=bnd_file)
tri = fset.get_mtri_triangulation()

# get velocity at final time
ux = fset.get_field(1, time)
uy = fset.get_field(2, time)
velocity = np.sqrt(ux**2 + uy**2)

Plot result :



In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9.2, 5.))
cylag.set_rcparams()
color,_,_ = cylag.get_default_color_palette()
ax.set_aspect('equal')

# plot mesh and velocity
levels = np.linspace(0.05, 0.15, 11)
cs = ax.tricontourf(tri, velocity, levels=levels, cmap='RdBu_r', alpha=0.80)
fig.colorbar(cs, ax=ax, label="$\\textbf{U}_f$ (m/s)")
ax.triplot(tri, lw=0.12, c='grey')

# plot ini positions
ax.plot(part.xp[0][:], part.yp[0][:], c='b', lw=0., marker='o', markersize=1., label="$x_p(0)$")

# plot final positions
i1 = int(1.*rec/4.)
i2 = int(2.*rec/4.)
i3 = int(3.*rec/4.)
ax.plot(part.xp[i1][:], part.yp[i1][:], c=color[4], lw=0., marker='o', markersize=1., label="$x_p(t_f/4)$")
ax.plot(part.xp[i2][:], part.yp[i2][:], c=color[1], lw=0., marker='o', markersize=1., label="$x_p(2t_f/4)$")
ax.plot(part.xp[i3][:], part.yp[i3][:], c=color[2], lw=0., marker='o', markersize=1., label="$x_p(3t_f/4)$")
ax.plot(part.xp[rec][:], part.yp[rec][:], c=color[0], lw=0., marker='o', markersize=1., label="$x_p(t_f)$")

plt.xlabel("$x$ (m)")
plt.ylabel("$y$ (m)")
plt.legend()
plt.savefig("figs/cylag_meander_2d_lsm1_keps.png", dpi=300, format="png")
plt.show()

Clean:



In [ ]:
del fset
del pset
del parameters
del cylag_solver
del part
del rec